# Free MuseTalk 1.5 lip-sync test (Kaggle GPU)

This private preview runs the open-source MuseTalk 1.5 model on Kaggle's free NVIDIA T4 GPU pool. GitHub Actions creates a private Kaggle Dataset with one existing Kling presenter clip and one matching narration track, then submits this notebook as a private GPU kernel.

The repository workflow uses the GitHub Actions secret `KAGGLE_API_TOKEN` and discovers the Kaggle account from the token. The workflow is manually triggerable and can also run the first preview after its narrowly scoped merge trigger. It never publishes to YouTube.

**Budget: USD 0.** The submission explicitly requests Kaggle's `NvidiaTeslaT4` free GPU pool. If Kaggle cannot attach a GPU, the first cell stops immediately; no CPU fallback and no paid service are used. Keep the first preview short (about 4–8 seconds). Do not put credentials in the dataset.


In [ ]:
import glob, os, shutil, subprocess, torch

print("Kaggle base-image PyTorch:", torch.__version__)
print("Base-image CUDA visibility:", torch.cuda.is_available())

# Some Kaggle base images omit the nvidia-smi binary. Check device nodes instead,
# then validate real CUDA access after installing the dedicated MuseTalk CUDA wheel.
gpu_nodes = sorted(glob.glob("/dev/nvidia*"))
print("NVIDIA device nodes:", gpu_nodes)
if not any(os.path.basename(p).startswith("nvidia") and p.endswith(("0", "ctl")) for p in gpu_nodes):
    raise RuntimeError(
        "Kaggle accepted the GPU notebook metadata but did not mount an NVIDIA device. "
        "Free GPU quota/capacity may be unavailable. Stop here; do not use CPU or paid fallback."
    )

nvidia_smi = shutil.which("nvidia-smi")
if nvidia_smi:
    probe = subprocess.run([nvidia_smi, "-L"], capture_output=True, text=True)
    print("nvidia-smi:", probe.stdout.strip() or probe.stderr.strip())
else:
    print("nvidia-smi executable is absent; the CUDA-enabled MuseTalk venv will verify the GPU in the install cell.")


In [ ]:
%%bash
set -euo pipefail
apt-get update -qq
apt-get install -y -qq ffmpeg
git clone https://github.com/TMElyralab/MuseTalk.git /kaggle/working/MuseTalk
cd /kaggle/working/MuseTalk

# Upstream MuseTalk pins NumPy/TensorFlow to versions that do not support
# Kaggle's current Python 3.13 image. Use the project's recommended Python 3.10.
python -m pip install -q --upgrade uv
uv python install 3.10
# Seed the environment with pip/setuptools/wheel. uv otherwise creates a
# pip-free environment, so "python -m pip" would fail on the next line.
uv venv --seed /kaggle/working/musetalk-venv --python 3.10
MUSEPY=/kaggle/working/musetalk-venv/bin/python
MUSEBIN=/kaggle/working/musetalk-venv/bin
"$MUSEPY" -m pip --version
"$MUSEPY" -m pip install -q --upgrade pip setuptools wheel

# MuseTalk's documented CUDA 11.8 build supports Kaggle's NVIDIA T4 pool.
"$MUSEPY" -m pip install -q torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 \
  --index-url https://download.pytorch.org/whl/cu118
"$MUSEPY" -m pip install -q -r requirements.txt
# The upstream MuseTalk checkout contains source files under musetalk/whisper,
# but that folder has no setup.py or pyproject.toml. Inference uses the
# Transformers WhisperModel and does not require pip-installing that folder.

# Install the OpenMMLab runtime components required by MuseTalk's DWPose
# preprocessing. These match the upstream MuseTalk installation guidance.
"$MUSEPY" -m pip install -q --no-cache-dir -U openmim
"$MUSEBIN/mim" install -q mmengine
"$MUSEBIN/mim" install -q "mmcv==2.0.1"
"$MUSEBIN/mim" install -q "mmdet==3.1.0"
# MMPose declares chumpy (legacy SMPL fitting) as a setup dependency; MuseTalk DWPose inference does not use it.
# Install MMPose without optional/legacy deps to avoid chumpy failing to build on modern pip.
"$MUSEPY" -m pip install -q --no-deps "mmpose==1.1.0"
"$MUSEPY" -m pip install -q json_tricks munkres xtcocotools yapf
"$MUSEPY" -m pip install -q huggingface_hub==0.30.2 gdown

# MuseTalk's preprocessing imports its bundled face_detection package by
# its top-level module name; expose that source folder as well as the repo root.
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
# Kaggle sets an IPython-only Matplotlib backend. MuseTalk runs in a separate
# Python 3.10 environment, so force a headless backend before importing MMPose.
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"

# Fail here with a clear message instead of later during video inference.
"$MUSEPY" - <<'PY'
import torch, matplotlib
if matplotlib.get_backend().lower() != "agg":
    raise RuntimeError(f"Expected Matplotlib Agg backend, got {matplotlib.get_backend()}")
print("Matplotlib backend:", matplotlib.get_backend())
import mmcv, mmdet, mmpose
from transformers import WhisperModel
from musetalk.utils.audio_processor import AudioProcessor
from face_detection import FaceAlignment, LandmarksType
from mmpose.apis import inference_topdown, init_model

print("MuseTalk PyTorch:", torch.__version__)
print("MuseTalk CUDA:", torch.cuda.is_available())
print("MuseTalk GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")
print("mmcv:", mmcv.__version__)
print("mmdet:", mmdet.__version__)
print("mmpose:", mmpose.__version__)
print("WhisperModel, AudioProcessor and face_detection imports: OK")
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
PY

# Prove the dedicated Python environment can see the requested GPU.
"$MUSEPY" - <<'PY'
import torch
print("MuseTalk environment PyTorch:", torch.__version__)
print("MuseTalk environment CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("MuseTalk Python 3.10 environment cannot see the free Kaggle GPU.")
print("MuseTalk GPU:", torch.cuda.get_device_name(0))
PY

# The upstream downloader otherwise upgrades huggingface_hub without a pin.
python - <<'PY'
from pathlib import Path
p = Path("download_weights.sh")
s = p.read_text()
s = s.replace(
    'pip install -U "huggingface_hub[cli]"',
    'pip install "huggingface_hub[cli]==0.30.2"'
)
p.write_text(s)
PY

echo "MuseTalk source and Python 3.10 CUDA dependencies installed."


In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk
source /kaggle/working/musetalk-venv/bin/activate
bash download_weights.sh
echo "Checkpoint download finished."


In [ ]:
# Find one video and one matching audio file in the attached private Kaggle Dataset.
from pathlib import Path
import os, glob, shutil
video_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.mp4','.mov','.mkv'))]
audio_candidates = [p for p in glob.glob('/kaggle/input/**/*', recursive=True) if p.lower().endswith(('.wav','.mp3','.m4a'))]
print('Video files:', video_candidates)
print('Audio files:', audio_candidates)
if len(video_candidates) != 1 or len(audio_candidates) != 1:
    raise RuntimeError('Attach a private Kaggle Dataset containing exactly one test video and one matching narration audio file.')
work = Path('/kaggle/working/MuseTalk')
video_path = Path(video_candidates[0])
audio_path = Path(audio_candidates[0])
data_dir = work / 'data' / 'free_test'
data_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(video_path, data_dir / 'presenter.mp4')
shutil.copy2(audio_path, data_dir / ('narration' + audio_path.suffix))
print('Input staged:', list(data_dir.iterdir()))

In [ ]:
# Configure one inference task. MuseTalk recommends 25 fps input for best results.
import yaml
config = {
  'task_0': {
    'video_path': str(data_dir / 'presenter.mp4'),
    'audio_path': str(data_dir / ('narration' + audio_path.suffix)),
    'bbox_shift': 0
  }
}
config_path = work / 'configs' / 'inference' / 'kaggle_free_test.yaml'
config_path.parent.mkdir(parents=True, exist_ok=True)
with open(config_path, 'w') as f:
    yaml.safe_dump(config, f, sort_keys=False)
print(config_path.read_text())

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working/MuseTalk
export PYTHONPATH="$PWD/musetalk/utils:$PWD${PYTHONPATH:+:$PYTHONPATH}"
export MPLBACKEND=Agg
export MPLCONFIGDIR=/kaggle/working/matplotlib-config
mkdir -p "$MPLCONFIGDIR"
/kaggle/working/musetalk-venv/bin/python -m scripts.inference \
  --inference_config configs/inference/kaggle_free_test.yaml \
  --result_dir /kaggle/working/musetalk_preview \
  --unet_model_path models/musetalkV15/unet.pth \
  --unet_config models/musetalkV15/musetalk.json \
  --version v15 \
  --fps 25 \
  --batch_size 4 \
  --use_float16


In [ ]:
# Locate the rendered video and check its duration. Download it from the Kaggle output panel.
from pathlib import Path
from IPython.display import Video, display
import subprocess
outputs = list(Path('/kaggle/working/musetalk_preview').rglob('*.mp4'))
if not outputs:
    raise RuntimeError('MuseTalk did not produce an MP4. Inspect the inference cell logs; do not publish an unsynchronized fallback.')
for p in outputs:
    probe = subprocess.check_output(['ffprobe','-v','error','-show_entries','format=duration','-of','default=noprint_wrappers=1:nokey=1',str(p)], text=True).strip()
    print({'file': str(p), 'bytes': p.stat().st_size, 'duration_seconds': float(probe)})
    display(Video(str(p), embed=True))